# Singularity Analysis — NR cold start vs MLP warm start

In [ ]:
import sys
sys.path.insert(0, r"c:\Users\max\InverseKinematics")

SAVE_DATA  = "G:/My Drive/inverse_kinematics/hot_start"
SAVE_MODEL = f"{SAVE_DATA}/mlp_ik.pt"

NR_MAX_ITER = 50
NR_TOL      = 1e-3   # metres position tolerance
NR_DAMPING  = 1e-4   # damped least-squares lambda
N_TRAJ      = 60     # trajectory steps

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from ik.kinematics.fk import FK, FK_batch_full
from ik.data.dataset import IKDataset
from ik.model.mlp import MLP

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")

## Load model + scalers

In [ ]:
# Load train dataset only to get MinMax scalers (no test leakage)
train_ds = IKDataset("train", SAVE_DATA)
MinMax_X = train_ds.MinMax_X   # [min (15,), max (15,)]
MinMax_Y = train_ds.MinMax_Y   # [min (6,),  max (6,)]

model = MLP(input_dim=15).to(DEVICE)
model.load_state_dict(torch.load(SAVE_MODEL, map_location=DEVICE))
model.eval()
print("model loaded")

## Helpers — Jacobian, NR, MLP warm start

In [ ]:
def numerical_jacobian(q, eps=1e-6):
    """
    6×6 numerical Jacobian via finite differences.
    Rows 0:3 → rotation (axis-angle), rows 3:6 → position.
    """
    T0 = FK(q)
    J  = np.zeros((6, 6))
    for i in range(6):
        dq      = np.zeros(6); dq[i] = eps
        Ti      = FK(q + dq)
        J[3:, i] = (Ti[:3, 3] - T0[:3, 3]) / eps          # position
        dR       = Ti[:3, :3] @ T0[:3, :3].T
        J[:3, i] = np.array([dR[2,1]-dR[1,2],
                              dR[0,2]-dR[2,0],
                              dR[1,0]-dR[0,1]]) / (2 * eps)  # rotation
    return J


def manipulability(q):
    """Yoshikawa manipulability index: sqrt(det(J @ J.T)). Zero at singularity."""
    J = numerical_jacobian(q)
    return np.sqrt(max(np.linalg.det(J @ J.T), 0.0))


def log_SO3(R):
    """Matrix log of SO(3) → 3-vector (axis × angle)."""
    cos_t = np.clip((np.trace(R) - 1) / 2, -1, 1)
    theta = np.arccos(cos_t)
    if abs(theta) < 1e-10:
        return np.zeros(3)
    return theta / (2 * np.sin(theta)) * np.array([R[2,1]-R[1,2], R[0,2]-R[2,0], R[1,0]-R[0,1]])


def nr_ik(q_init, T_target, max_iter=NR_MAX_ITER, tol=NR_TOL, damping=NR_DAMPING):
    """
    Damped least-squares Newton-Raphson IK.
    Returns (q, iterations, converged).
    """
    q = q_init.copy()
    R_target = T_target[:3, :3]
    p_target = T_target[:3,  3]

    for it in range(max_iter):
        T   = FK(q)
        err = np.concatenate([
            log_SO3(R_target @ T[:3, :3].T),   # rotation error (3,)
            p_target - T[:3, 3]                 # position error (3,)
        ])

        if np.linalg.norm(err[3:]) < tol:      # convergence on position
            return q, it, True

        J  = numerical_jacobian(q)
        dq = J.T @ np.linalg.solve(J @ J.T + damping * np.eye(6), err)
        q  = q + dq

    return q, max_iter, False


def mlp_predict(q_current, T_target, model, MinMax_X, MinMax_Y, device):
    """
    One-shot MLP prediction of q_target given current config and target pose.
    Input: [R6_target(6), P_target(3), q_current(6)] = 15-dim, MinMax normalised.
    """
    R6 = T_target[:2, :3].flatten().astype(np.float32)  # first 2 rows of rotation
    P  = T_target[:3, 3].astype(np.float32)
    x  = np.concatenate([R6, P, q_current.astype(np.float32)])

    x_norm = (x - MinMax_X[0]) / (MinMax_X[1] - MinMax_X[0])
    x_norm = (x_norm * 2) - 1

    with torch.no_grad():
        x_t   = torch.tensor(x_norm, dtype=torch.float32).unsqueeze(0).to(device)
        y_t   = model(x_t).squeeze(0).cpu().numpy()

    # inverse MinMax
    q_pred = (y_t + 1) / 2 * (MinMax_Y[1] - MinMax_Y[0]) + MinMax_Y[0]
    return q_pred

## Map singularities — manipulability distribution

In [ ]:
from ik.kinematics.fk import JOINT_LIMITS

N_SAMPLES = 3000
np.random.seed(42)

q_samples = np.random.uniform(
    JOINT_LIMITS[:, 0], JOINT_LIMITS[:, 1], size=(N_SAMPLES, 6)
)
manip_vals = np.array([manipulability(q) for q in q_samples])

print(f"min:    {manip_vals.min():.4f}")
print(f"median: {np.median(manip_vals):.4f}")
print(f"max:    {manip_vals.max():.4f}")
print(f"near-singular (< 0.01): {(manip_vals < 0.01).sum()} / {N_SAMPLES}")

plt.figure(figsize=(8, 3))
plt.hist(manip_vals, bins=80, color="steelblue", edgecolor="none")
plt.axvline(0.01, color="red", linestyle="--", label="singularity threshold 0.01")
plt.xlabel("Manipulability")
plt.ylabel("Count")
plt.title("Manipulability distribution over random configs")
plt.legend()
plt.tight_layout()
plt.show()

## Wrist singularity trajectory

Joints 4 and 6 both rotate about the x-axis. When **joint 5 (wrist angle, index 4) → 0** they become collinear → wrist singularity.  
We build a trajectory where q[4] sweeps from +0.6 → 0 → -0.6 rad while all other joints are fixed.

In [ ]:
# Ground-truth joint configs along trajectory (what we want the robot to follow)
q_A = np.array([0.3, -0.4, 0.5, 0.2,  0.6, 0.3])   # wrist angle = +0.6 (away)
q_B = np.array([0.3, -0.4, 0.5, 0.2,  0.0, 0.3])   # wrist angle =  0.0 (singularity)
q_C = np.array([0.3, -0.4, 0.5, 0.2, -0.6, 0.3])   # wrist angle = -0.6 (away)

# Interpolate A→B→C in joint space to get ground-truth poses
t_AB   = np.linspace(0, 1, N_TRAJ // 2, endpoint=False)
t_BC   = np.linspace(0, 1, N_TRAJ // 2)
q_traj = np.vstack([
    q_A + t[:, None] * (q_B - q_A) for t in [t_AB]
] + [
    q_B + t[:, None] * (q_C - q_B) for t in [t_BC]
])  # (N_TRAJ, 6)

T_traj  = np.array([FK(q) for q in q_traj])           # (N_TRAJ, 4, 4) — target poses
m_traj  = np.array([manipulability(q) for q in q_traj])

# Plot EE path and manipulability along it
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
pos = T_traj[:, :3, 3]
sc  = ax.scatter(np.arange(N_TRAJ), pos[:, 2], c=m_traj, cmap="RdYlGn", s=20)
ax.set_xlabel("Step"); ax.set_ylabel("EE z (m)")
ax.set_title("EE z-position along trajectory")
plt.colorbar(sc, ax=ax, label="Manipulability")

ax = axes[1]
ax.plot(m_traj, color="steelblue")
ax.axhline(0.01, color="red", linestyle="--", label="singularity threshold")
ax.set_xlabel("Step"); ax.set_ylabel("Manipulability")
ax.set_title("Manipulability along trajectory")
ax.legend()

plt.tight_layout()
plt.show()
print(f"Min manipulability on trajectory: {m_traj.min():.5f} at step {m_traj.argmin()}")

## Run comparison — NR cold start vs NR MLP warm start

In [ ]:
cold_iters, cold_conv, cold_pos_err = [], [], []
warm_iters, warm_conv, warm_pos_err = [], [], []

q_cold = q_traj[0].copy()   # both start at the true first config
q_warm = q_traj[0].copy()

for i, T_target in enumerate(T_traj):
    # --- cold start: NR initialised from previous solution ---
    q_sol_cold, it_cold, conv_cold = nr_ik(q_cold, T_target)
    err_cold = np.linalg.norm(T_target[:3, 3] - FK(q_sol_cold)[:3, 3])

    # --- warm start: MLP prediction → NR ---
    q_mlp = mlp_predict(q_warm, T_target, model, MinMax_X, MinMax_Y, DEVICE)
    q_sol_warm, it_warm, conv_warm = nr_ik(q_mlp, T_target)
    err_warm = np.linalg.norm(T_target[:3, 3] - FK(q_sol_warm)[:3, 3])

    cold_iters.append(it_cold);   cold_conv.append(conv_cold);  cold_pos_err.append(err_cold)
    warm_iters.append(it_warm);   warm_conv.append(conv_warm);  warm_pos_err.append(err_warm)

    # chain: next step initialises from this step's solution
    q_cold = q_sol_cold
    q_warm = q_sol_warm   # warm: chain from NR solution (MLP is the init, not the chain)

cold_iters  = np.array(cold_iters);  cold_pos_err = np.array(cold_pos_err)
warm_iters  = np.array(warm_iters);  warm_pos_err = np.array(warm_pos_err)

print(f"{'':25s} {'cold':>10} {'warm':>10}")
print(f"{'mean iterations':25s} {cold_iters.mean():>10.2f} {warm_iters.mean():>10.2f}")
print(f"{'converged (%)':25s} {100*np.mean(cold_conv):>10.1f} {100*np.mean(warm_conv):>10.1f}")
print(f"{'mean pos error (m)':25s} {cold_pos_err.mean():>10.4f} {warm_pos_err.mean():>10.4f}")
print(f"{'max pos error (m)':25s} {cold_pos_err.max():>10.4f} {warm_pos_err.max():>10.4f}")

## Visualise results

In [ ]:
steps = np.arange(N_TRAJ)
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# — iterations to convergence —
ax = axes[0]
ax.plot(steps, cold_iters, label="NR cold (q_prev)", color="tomato",    linewidth=1.5)
ax.plot(steps, warm_iters, label="NR warm (MLP)",    color="steelblue", linewidth=1.5)
ax2 = ax.twinx()
ax2.fill_between(steps, m_traj, alpha=0.15, color="green", label="manipulability")
ax2.set_ylabel("Manipulability", color="green")
ax.set_ylabel("Iterations"); ax.set_title("Iterations to convergence")
ax.legend(loc="upper left"); ax.grid(True, alpha=0.3)

# — position error at convergence —
ax = axes[1]
ax.semilogy(steps, cold_pos_err * 1000, label="NR cold", color="tomato",    linewidth=1.5)
ax.semilogy(steps, warm_pos_err * 1000, label="NR warm", color="steelblue", linewidth=1.5)
ax.axhline(NR_TOL * 1000, color="gray", linestyle="--", label=f"tol = {NR_TOL*1000:.1f} mm")
ax.set_ylabel("Pos error (mm)"); ax.set_title("Position error at convergence")
ax.legend(); ax.grid(True, alpha=0.3)

# — manipulability along trajectory —
ax = axes[2]
ax.plot(steps, m_traj, color="seagreen", linewidth=1.5)
ax.axhline(0.01, color="red", linestyle="--", label="singularity threshold 0.01")
singular_steps = np.where(~np.array(cold_conv))[0]
warm_fail_steps = np.where(~np.array(warm_conv))[0]
if len(singular_steps):
    ax.scatter(singular_steps, m_traj[singular_steps], color="tomato",    zorder=5, label="cold failed")
if len(warm_fail_steps):
    ax.scatter(warm_fail_steps, m_traj[warm_fail_steps], color="steelblue", zorder=5, label="warm failed", marker="x", s=60)
ax.set_xlabel("Trajectory step"); ax.set_ylabel("Manipulability")
ax.set_title("Manipulability along trajectory"); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()